In [2]:
import gymnasium as gym
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
import numpy as np
from collections import deque
import random
import time
import imageio

2026-02-24 16:42:01.305556: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
env = gym.make('LunarLander-v3', render_mode='rgb_array')

state_size = env.observation_space.shape[0]
num_actions = int(env.action_space.n)
print(f"State Vector Size: {state_size}")
print(f"Action Space Size: {num_actions}")

State Vector Size: 8
Action Space Size: 4


In [4]:
# Defining the neural network architecture
def build_q_network(state_size, num_actions):
    model = Sequential([
        Input(shape=(state_size,)),
        Dense(64, activation='relu'),
        Dense(64, activation='relu'),
        Dense(num_actions, activation='linear')
    ])
    return model

In [5]:
# Creating Dual network system
main_q_network = build_q_network(state_size, num_actions)
target_q_network = build_q_network(state_size, num_actions)

# Initialize the target network with same weights as of main network
target_q_network.set_weights(main_q_network.get_weights())

I0000 00:00:1771951339.627005    1956 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
optimizer = Adam(learning_rate=1e-3)
main_q_network.compile(loss='mse', optimizer=optimizer)

print("Environment and Dual-Networks successfully initialized.")

Environment and Dual-Networks successfully initialized.


Replay buffer to keep the data to train the model 

In [7]:
memory_buffer = deque(maxlen=100000)
def store_experience(state, action, reward, next_state, done):
    memory_buffer.append((state, action, reward, next_state, done))

Implementing the greedy policy 

In [8]:
def choose_action(state, epsilon, main_q_network, num_actions):
    if np.random.rand() < epsilon:
        return np.random.choice(num_actions)

    else:
        state_tensor = np.expand_dims(state, axis=0)
        q_values = main_q_network.predict(state_tensor, verbose=0)

        return np.argmax(q_values[0])

Bellman update function

In [9]:
GAMMA = 0.99
BATCH_SIZE = 64

@tf.function 

def agent_learn(experiences, gamma):
    states, actions, rewards, next_states, dones = experiences

    next_q_values = target_q_network(next_states)
    max_next_q_values = tf.reduce_max(next_q_values, axis=1)

    target_q_values = rewards + (gamma * max_next_q_values * (1 - dones))

    with tf.GradientTape() as tape:
        all_q_values = main_q_network(states)

        actions = tf.cast(actions, tf.int32)
        masks = tf.one_hot(actions, num_actions)
        q_values = tf.reduce_sum(tf.multiply(all_q_values, masks), axis=1)

        loss = tf.keras.losses.MSE(target_q_values, q_values)

    gradients = tape.gradient(loss, main_q_network.trainable_variables)
    optimizer.apply_gradients(zip(gradients, main_q_network.trainable_variables))

Supporting Extraction function

In [10]:
def sample_experiences(memory_buffer, batch_size):
    experiences = random.sample(memory_buffer, k=batch_size)

    states = tf.convert_to_tensor([e[0] for e in experiences], dtype=tf.float32)
    actions = tf.convert_to_tensor([e[1] for e in experiences], dtype=tf.float32)
    rewards = tf.convert_to_tensor([e[2] for e in experiences], dtype=tf.float32)
    next_states = tf.convert_to_tensor([e[3] for e in experiences], dtype=tf.float32)

    dones = tf.convert_to_tensor([e[4] for e in experiences], dtype=tf.float32)

    return (states, actions, rewards, next_states, dones)

The execution loop for the model

In [12]:
NUM_EPISODES = 2000
MAX_STEPS_PER_EPISODE = 1000
TARGET_UPDATE_FREQ =100
MEMORY_MIN_SIZE = 64

epsilon = 1.0
EPSILON_DECAY = 0.995
EPSILON_MIN = 0.01

total_steps = 0
reward_history = []

print("Initiating the Lunar lander Training Sequence..")
start_time = time.time()

for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    episode_reward = 0

    for step in range(MAX_STEPS_PER_EPISODE):
        action = choose_action(state, epsilon, main_q_network, num_actions)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        store_experience(state, action, reward, next_state, done)

        if len(memory_buffer) > MEMORY_MIN_SIZE:
            experiences = sample_experiences(memory_buffer, BATCH_SIZE)
            agent_learn(experiences, GAMMA)

        if total_steps % TARGET_UPDATE_FREQ == 0:
            target_q_network.set_weights(main_q_network.get_weights())

        state = next_state
        episode_reward += reward
        total_steps += 1

        if done:
            break

    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

    reward_history.append(episode_reward)
    recent_average = np.mean(reward_history[-100:])

    if (episode + 1) % 10 == 0:
        print(f"Episode: {episode + 1} | Avg Reward (Last 100): {recent_average:.2f} | Epsilon: {epsilon:.2f}")
        
    if recent_average >= 200.0:
        print(f"\nEnvironment solved in {episode + 1} episodes!")
        print(f"Final Average Reward: {recent_average:.2f}")
        main_q_network.save('lunar_lander_dqn_model.h5') 
        break

end_time = time.time()
print(f"Training completed in {(end_time - start_time) / 60:.2f} minutes.")

Initiating the Lunar lander Training Sequence..


2026-02-24 10:45:06.484481: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d59f2d78630 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-24 10:45:06.484530: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-02-24 10:45:06.500814: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-24 10:45:06.542345: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
I0000 00:00:1771929906.721639    5854 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Episode: 10 | Avg Reward (Last 100): -193.49 | Epsilon: 0.95
Episode: 20 | Avg Reward (Last 100): -181.91 | Epsilon: 0.90
Episode: 30 | Avg Reward (Last 100): -172.70 | Epsilon: 0.86
Episode: 40 | Avg Reward (Last 100): -170.85 | Epsilon: 0.82
Episode: 50 | Avg Reward (Last 100): -158.66 | Epsilon: 0.78
Episode: 60 | Avg Reward (Last 100): -147.36 | Epsilon: 0.74
Episode: 70 | Avg Reward (Last 100): -136.11 | Epsilon: 0.70
Episode: 80 | Avg Reward (Last 100): -131.20 | Epsilon: 0.67
Episode: 90 | Avg Reward (Last 100): -123.54 | Epsilon: 0.64
Episode: 100 | Avg Reward (Last 100): -122.67 | Epsilon: 0.61
Episode: 110 | Avg Reward (Last 100): -107.11 | Epsilon: 0.58
Episode: 120 | Avg Reward (Last 100): -97.32 | Epsilon: 0.55
Episode: 130 | Avg Reward (Last 100): -91.60 | Epsilon: 0.52
Episode: 140 | Avg Reward (Last 100): -79.68 | Epsilon: 0.50
Episode: 150 | Avg Reward (Last 100): -75.97 | Epsilon: 0.47
Episode: 160 | Avg Reward (Last 100): -71.07 | Epsilon: 0.45
Episode: 170 | Avg Rew

KeyboardInterrupt: 

In [11]:
main_q_network.save('lunar_lander_dqn_model_score96.h5')
print("Model saved successfully.")

Model saved successfully.


In [13]:
vis_env = gym.make('LunarLander-v3', render_mode='rgb_array')

print("Recording Lunar Lander flight...")

state, _ = vis_env.reset()
frames = []
total_reward = 0

for step in range(1000):
    frame = vis_env.render()
    frames.append(frame)
    
    state_tensor = np.expand_dims(state, axis=0)
    q_values = main_q_network.predict(state_tensor, verbose=0)
    action = np.argmax(q_values[0])
    next_state, reward, terminated, truncated, _ = vis_env.step(action)
    
    state = next_state
    total_reward += reward
    
    if terminated or truncated:
        break
vis_env.close()
# Save the frames as an MP4 video
video_filename = 'trained_lunar_lander.mp4'
imageio.mimsave(video_filename, frames, fps=30)
print(f"Flight recording complete! Video saved as: {video_filename}")
print(f"Flight Score: {total_reward:.2f}")

Recording Lunar Lander flight...


Flight recording complete! Video saved as: trained_lunar_lander.mp4
Flight Score: -397.96
